# Test retrieval (vector search -> expand Article -> preamble -> group by Article)

Gọi thẳng `retrieval.vector_retriever.retrieve_relevant_clauses()` (không qua LLM evaluate) để xem
TOÀN BỘ kết quả truy hồi: Excerpt nào khớp, điểm số, Clause nào được kéo thêm theo Điều, preamble
có được gắn không. Khác `pipeline_test.ipynb` (chạy hết pipeline tới LLM), notebook này dừng ở
bước retrieval để debug riêng phần này. Output in FULL (không cắt chuỗi `text[:200]`).

Yêu cầu: contract đã được import + build graph trước đó (contract_id bên dưới phải tồn tại trong
Neo4j - chạy `pipeline_test.ipynb` hoặc `import_flow.ipynb` trước nếu chưa có).

In [1]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

CONTRACT_ID = 1


def print_full(obj):
    """In FULL, không cắt chuỗi, không rút gọn dict/list (mặc định repr/print của Python/pandas
    hay cắt bớt text dài)."""
    print(json.dumps(obj, ensure_ascii=False, indent=2))

## 1. Retrieval cho 1 câu hỏi checklist đơn

In [4]:
from rag.retrieval.vector_retriever import retrieve_relevant_clauses

item = {
    "question": "Mức phạt vi phạm hợp đồng là bao nhiêu?",
    "pass_criteria": "Có quy định rõ mức phạt (%) và tổng mức phạt tối đa không vượt quá 8% giá trị phần nghĩa vụ vi phạm theo quy định pháp luật.",
    "violation_criteria": "Không quy định mức phạt, hoặc mức phạt tối đa vượt quá 8% giá trị nghĩa vụ vi phạm.",
}

groups = retrieve_relevant_clauses(CONTRACT_ID, item)

n_clauses = sum(len(g["clauses"]) for g in groups)
print(f"{len(groups)} Dieu, {n_clauses} Khoan\n")
print_full(groups)

3 Dieu, 9 Khoan

[
  {
    "article_number": "0",
    "article_title": "Thông tin chung và các bên tham gia hợp đồng",
    "score": null,
    "clauses": [
      {
        "number": "0",
        "title": "Thông tin chung và các bên tham gia hợp đồng",
        "article_number": "0",
        "article_title": "Thông tin chung và các bên tham gia hợp đồng",
        "score": null,
        "text": "## CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc\n\n## HỢP ĐỒNG DỊCH VỤ PHÁT TRIỂN PHẦN MỀM\n\nSố: 123/VIVA-VTIT/2026\n\nHôm nay, ngày 19 tháng 3 năm 2026, tại trụ sở làm việc của Khách hàng tại số 36A Dịch Vọng Hậu, phường Cầu Giấy, TP Hà Nội, chúng tôi gồm:\n\n## BÊN SỬ DỤNG DỊCH VỤ: CÔNG TY TNHH VIVAPLAST\n\nĐịa chỉ : 36B Dịch Vọng Hậu, phường Cầu Giấy, Hà Nội, Việt Nam\n\nMã số thuế : 85415419613\n\nSĐT : 033444555\n\nĐại diện: Ông Nguyễn Văn A\n\nChức vụ : Phó Giám đốc\n\n(Sau đây gọi chung là ' Khách hàng ')\n\n## BÊN CUNG CẤP DỊCH VỤ: CÔNG TY TRÁCH NHIỆM HỮU HẠN MỘT THÀNH VI

## 2. Chỉ xem phần text ghép sẵn của từng Điều (giống context sẽ đưa vào LLM)

In [3]:
for g in groups:
    score = f"{g['score']:.4f}" if g["score"] is not None else "-"
    print(f"===== Dieu {g['article_number']} (score={score}) =====")
    print(g["text"])
    print()

===== Dieu 0 (score=-) =====
Điều 0. Thông tin chung và các bên tham gia hợp đồng

## CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc lập - Tự do - Hạnh phúc

## HỢP ĐỒNG DỊCH VỤ PHÁT TRIỂN PHẦN MỀM

Số: 123/VIVA-VTIT/2026

Hôm nay, ngày 19 tháng 3 năm 2026, tại trụ sở làm việc của Khách hàng tại số 36A Dịch Vọng Hậu, phường Cầu Giấy, TP Hà Nội, chúng tôi gồm:

## BÊN SỬ DỤNG DỊCH VỤ: CÔNG TY TNHH VIVAPLAST

Địa chỉ : 36B Dịch Vọng Hậu, phường Cầu Giấy, Hà Nội, Việt Nam

Mã số thuế : 85415419613

SĐT : 033444555

Đại diện: Ông Nguyễn Văn A

Chức vụ : Phó Giám đốc

(Sau đây gọi chung là ' Khách hàng ')

## BÊN CUNG CẤP DỊCH VỤ: CÔNG TY TRÁCH NHIỆM HỮU HẠN MỘT THÀNH VIÊN ĐẦU TƯ CÔNG NGHỆ VIETTEL

Địa chỉ : Nam Đại lộ Lê Lợi, phường Hạc Thành, tỉnh Thanh Hóa, Việt Nam

Mã số thuế : 2801045888

Đại diện : Ông Bùi Trình

Chức vụ : Giám đốc

(Sau đây gọi chung là ' VTIT ')

(Khách hàng và VTIT sau đây có thể được gọi riêng là 'Bên' và gọi chung là ' Các Bên ' hoặc ' Hai Bên ')

## Xét rằng:

- -Khách

## 3. Chạy nhiều câu hỏi liên tiếp, in full kết quả từng câu

In [ ]:
SAMPLE_ITEMS = [
    {
        "question": "Người đại diện ký hợp đồng đã xác định đúng theo phân cấp, ủy quyền chưa?",
        "pass_criteria": "Người ký là Giám đốc/Tổng Giám đốc hoặc có giấy ủy quyền hợp lệ.",
        "violation_criteria": "Người ký không có thẩm quyền, không có ủy quyền.",
    },
    {
        "question": "Hợp đồng có điều khoản bảo mật thông tin không?",
        "pass_criteria": "",
        "violation_criteria": "",
    },
]

for it in SAMPLE_ITEMS:
    result = retrieve_relevant_clauses(CONTRACT_ID, it)
    print(f"### Cau hoi: {it['question']}")
    print_full(result)
    print("=" * 100)
    print()